# Colab 7B Experiments (Hugging Face GPU)

This notebook is for the 7B model runs you own. It runs models directly on the Colab GPU with Hugging Face `transformers` and 4-bit `bitsandbytes` quantization. It does **not** use Ollama.

Scope:
- Model order: Qwen 7B first, then LLaMA 7B, then Gemma 7B.
- Datasets match the merged `feature/icl-strategies` loaders: `gsm_symbolic`, `gsm_plus`, `gsm_ic`, `bigbench_hard`, `bigbench_hard_tracking`, `gsm8k`, and `folio`.
- Conditions match the merged runner: `zero_shot_baseline`, `zero_shot`, `targeted_few_shot_answer_only`, `random_few_shot`, `targeted_few_shot`, `targeted_few_shot_k5`, `error_targeted_icl`, `error_targeted_icl_random`, `error_targeted_icl_correct_only`, and `self_consistency`.
- Outputs are copied to a separate Google Drive output folder after each completed combination, with logs, manifest files, and a summary CSV kept outside the code path.

Important limitations:
- The merged repo still uses an Ollama `models/model_loader.py`; this notebook patches only the Drive runtime copy to use Hugging Face/Transformers for Colab.
- LLaMA and Gemma model IDs on Hugging Face usually require accepting the model license and using an HF token. Add your token as a Colab secret named `HF_TOKEN` before running those.
- Full runs are large: 3 models x 7 datasets x 10 conditions by default, and `self_consistency` generates multiple responses per question. Keep `RUN_FULL_EXPERIMENTS = False` until smoke checks pass.
- The repo saves incrementally inside each subprocess to `results/raw/<benchmark>/...`. This notebook copies completed raw files to `OUTPUT_ROOT/raw` and skips completed manifest combinations on resume.

## Run Order

1. Run the setup cells through all smoke tests first.
2. Confirm the CLI smoke run writes a result JSON and the summary cell can read it.
3. Set `RUN_FULL_EXPERIMENTS = True` only when you are ready to spend Colab credits.
4. If Colab disconnects, reconnect, rerun setup/smoke cells as needed, then rerun the full-run cell. Successful combinations recorded in the manifest are skipped unless `FORCE_RERUN = True`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

In [ ]:
CODEBASE_DIR = '/content/drive/MyDrive/COMP6242/Project/llm-failure'  # folder containing run_experiment.py
OUTPUT_ROOT = '/content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results'  # persistent experiment outputs
MODEL_CACHE_DIR = f'{OUTPUT_ROOT}/hf-cache'  # persistent HF model/dataset cache on Drive

# Optional. Prefer adding HF_TOKEN as a Colab secret instead of pasting it here.
HF_TOKEN = None

# Safety switches.
RUN_FULL_EXPERIMENTS = True  # full run enabled after smoke test passed
FORCE_RERUN = False
REQUIRE_ALL_DATASETS = True  # FOLIO access is approved, so fail fast if any selected dataset is unavailable

# Keep the default sample count modest for Colab. Increase only if the group agreed.
FULL_NUM_SAMPLES = 200
SMOKE_TEST_SAMPLES = 2
FEW_SHOT_POOL_SIZE = 20
NUM_EXAMPLES = 3
NUM_EXAMPLES_K5 = 5
SELF_CONSISTENCY_SAMPLES = 5
SELF_CONSISTENCY_TEMPERATURE = 0.7
MAX_TOKENS_PER_RESPONSE = 1024
LOAD_IN_4BIT = True

# Implemented in data/data_loader.py after merging feature/icl-strategies.
DATASETS = [
    'gsm_symbolic',
    'gsm_plus',
    'gsm_ic',
    'bigbench_hard',
    'bigbench_hard_tracking',
    'gsm8k',
    'folio',
]

CONDITIONS = [
    'zero_shot_baseline',
    'zero_shot',
    'targeted_few_shot_answer_only',
    'random_few_shot',
    'targeted_few_shot',
    'targeted_few_shot_k5',
    'error_targeted_icl',
    'error_targeted_icl_random',
    'error_targeted_icl_correct_only',
    'self_consistency',
]

# Ordered plan: Qwen first, then LLaMA, then Gemma.
# LLaMA/Gemma usually need a Hugging Face token with license access.
MODEL_PLAN = [
    {
        'family': 'qwen',
        'display_name': 'Qwen/Qwen2.5-7B-Instruct',
        'hf_model_id': 'Qwen/Qwen2.5-7B-Instruct',
        'enabled': True,
    },
    # {
    #     'family': 'llama',
    #     'display_name': 'meta-llama/Llama-2-7b-chat-hf',
    #     'hf_model_id': 'meta-llama/Llama-2-7b-chat-hf',
    #     'enabled': True,
    # },
    {
        'family': 'gemma',
        'display_name': 'google/gemma-7b-it',
        'hf_model_id': 'google/gemma-7b-it',
        'enabled': True,
    },
]

PRIMARY_SMOKE_FAMILY = 'qwen'
SMOKE_BENCHMARK = 'gsm_symbolic'
SMOKE_CONDITION = 'zero_shot'

print('Enabled model order:', [m['family'] for m in MODEL_PLAN if m['enabled']])
print('Datasets:', DATASETS)
print('Conditions:', CONDITIONS)

In [ ]:
from pathlib import Path
import os
import sys

CODEBASE = Path(CODEBASE_DIR).expanduser()
OUTPUT = Path(OUTPUT_ROOT).expanduser()
MODEL_CACHE = Path(MODEL_CACHE_DIR).expanduser()

if not CODEBASE.exists():
    raise FileNotFoundError(f'CODEBASE_DIR does not exist: {CODEBASE}')
if not (CODEBASE / 'run_experiment.py').exists():
    raise FileNotFoundError(f'run_experiment.py not found under CODEBASE_DIR: {CODEBASE}')
if CODEBASE.resolve() == OUTPUT.resolve():
    raise ValueError('CODEBASE_DIR and OUTPUT_ROOT must be different folders.')

OUTPUT_RAW_DIR = OUTPUT / 'raw'
OUTPUT_LOG_DIR = OUTPUT / 'logs'
OUTPUT_STATUS_DIR = OUTPUT / 'status'
REPO_RESULTS_DIR = CODEBASE / 'results' / 'raw'

for path in [OUTPUT, OUTPUT_RAW_DIR, OUTPUT_LOG_DIR, OUTPUT_STATUS_DIR, REPO_RESULTS_DIR, MODEL_CACHE]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(MODEL_CACHE)
os.environ['HF_HUB_CACHE'] = str(MODEL_CACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(MODEL_CACHE / 'transformers')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

os.chdir(CODEBASE)
if str(CODEBASE) not in sys.path:
    sys.path.insert(0, str(CODEBASE))

print('Codebase:', CODEBASE)
print('Persistent outputs:', OUTPUT)
print('Repo raw results:', REPO_RESULTS_DIR)
print('HF cache:', MODEL_CACHE)

In [ ]:
# Install the Colab GPU inference stack directly.
# Avoid upgrading NumPy/SciPy blindly: Colab preinstalls binary packages that can break if NumPy moves too far ahead.

import subprocess
import sys

base_stack = [
    'numpy==2.0.2',
    'scipy==1.14.1',
    'scikit-learn==1.6.1',
    'pandas',
]
inference_stack = [
    'transformers>=4.45.0',
    'accelerate',
    'bitsandbytes',
    'datasets',
    'huggingface_hub',
    'sentencepiece',
    'protobuf',
    'tqdm',
    'matplotlib',
    'seaborn',
]

for packages in [base_stack, inference_stack]:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *packages]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('Dependency install step finished.')
print('If you already ran the older dependency cell in this runtime, restart the Colab runtime once, then rerun from the Drive mount cell.')

Running: /usr/bin/python3 -m pip install -q numpy==2.0.2 scipy==1.14.1 scikit-learn==1.6.1 pandas
Running: /usr/bin/python3 -m pip install -q transformers>=4.45.0 accelerate bitsandbytes datasets huggingface_hub sentencepiece protobuf tqdm matplotlib seaborn
Dependency install step finished.
If you already ran the older dependency cell in this runtime, restart the Colab runtime once, then rerun from the Drive mount cell.


In [ ]:
# Runtime repo preparation for direct Hugging Face inference.
# This patches the Drive copy of config.py and models/model_loader.py so the existing CLI can run qwen/llama/gemma families on the Colab GPU.

import pprint
import re
from pathlib import Path

models_by_family = {
    item['family']: [item['display_name']]
    for item in MODEL_PLAN
    if item['enabled']
}
model_name_map = {
    item['display_name']: item['hf_model_id']
    for item in MODEL_PLAN
    if item['enabled']
}

config_path = CODEBASE / 'config.py'
loader_path = CODEBASE / 'models' / 'model_loader.py'
backup_path = CODEBASE / 'models' / 'model_loader_ollama_backup.py'

if loader_path.exists() and not backup_path.exists():
    backup_path.write_text(loader_path.read_text(encoding='utf-8'), encoding='utf-8')

config_text = config_path.read_text(encoding='utf-8')
# Remove any broken commented MODELS block left by an earlier notebook run.
config_text = re.sub(r"(?m)^#\s*MODELS\s*=.*(?:\n\s+['\"][^\n]*)*\n?", "", config_text)

def replace_assignment(text, name, value_repr):
    pattern = rf'^(?!\s*#)\s*{re.escape(name)}\s*=\s*.*$'
    replacement = f'{name} = {value_repr}'
    new_text, count = re.subn(pattern, replacement, text, flags=re.MULTILINE)
    if count == 0:
        new_text = text.rstrip() + '\n' + replacement + '\n'
    return new_text

def replace_literal_block(text, name, value):
    pattern = rf'(?m)^(?!\s*#)\s*{re.escape(name)}\s*=\s*(\{{|\[)'
    match = re.search(pattern, text)
    replacement = f'{name} = {pprint.pformat(value, sort_dicts=False)}'
    if not match:
        return text.rstrip() + f'\n{replacement}\n'

    start = match.start()
    opener = match.group(1)
    closer = '}' if opener == '{' else ']'
    depth = 0
    end = None
    for idx in range(match.start(1), len(text)):
        char = text[idx]
        if char == opener:
            depth += 1
        elif char == closer:
            depth -= 1
            if depth == 0:
                end = idx + 1
                break
    if end is None:
        raise ValueError(f'Could not find end of {name} block in config.py')

    return text[:start] + replacement + text[end:]

config_text = replace_assignment(config_text, 'SMOKE_TEST', 'False')
config_text = replace_assignment(config_text, 'SMOKE_TEST_SAMPLES', str(SMOKE_TEST_SAMPLES))
config_text = replace_assignment(config_text, 'NUM_SAMPLES', str(FULL_NUM_SAMPLES))
config_text = replace_assignment(config_text, 'MAX_NEW_TOKENS', str(MAX_TOKENS_PER_RESPONSE))
config_text = replace_assignment(config_text, 'FEW_SHOT_POOL_SIZE', str(FEW_SHOT_POOL_SIZE))
config_text = replace_assignment(config_text, 'NUM_EXAMPLES', str(NUM_EXAMPLES))
config_text = replace_assignment(config_text, 'NUM_EXAMPLES_K5', str(NUM_EXAMPLES_K5))
config_text = replace_assignment(config_text, 'N_SAMPLES_S6', str(SELF_CONSISTENCY_SAMPLES))
config_text = replace_assignment(config_text, 'TEMPERATURE_S6', str(SELF_CONSISTENCY_TEMPERATURE))
config_text = replace_assignment(config_text, 'RESULTS_DIR', repr(str(REPO_RESULTS_DIR)))
config_text = replace_literal_block(config_text, 'MODELS', models_by_family)
config_text = replace_literal_block(config_text, 'BENCHMARKS', DATASETS)
config_text = replace_literal_block(config_text, 'CONDITIONS', CONDITIONS)
config_path.write_text(config_text, encoding='utf-8')

loader_template = r'''
import gc
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME_MAP = __MODEL_NAME_MAP__
LOAD_IN_4BIT = __LOAD_IN_4BIT__


def _compute_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def _resolve_model_id(model_name: str) -> str:
    return MODEL_NAME_MAP.get(model_name, model_name)


def load_model(model_name: str):
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU is not available. In Colab, choose Runtime > Change runtime type > GPU.')

    model_id = _resolve_model_id(model_name)
    token = os.environ.get('HF_TOKEN') or None
    dtype = _compute_dtype()

    print(f'Loading HF model: {model_id}')
    print(f'Quantization: {"4-bit" if LOAD_IN_4BIT else "none"}; dtype: {dtype}')

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=token,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )

    model_kwargs = {
        'token': token,
        'device_map': 'auto',
        'torch_dtype': dtype,
        'trust_remote_code': True,
        'low_cpu_mem_usage': True,
    }
    if quantization_config is not None:
        model_kwargs['quantization_config'] = quantization_config

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()

    return {'model': model, 'tokenizer': tokenizer, 'model_id': model_id}, tokenizer


def generate_response(model_bundle, promptText, maxTokens: int = __MAX_TOKENS__, temperature: float = 0) -> str:
    model = model_bundle['model']
    tokenizer = model_bundle['tokenizer']
    model_id = model_bundle['model_id']

    prompt = promptText
    if 'qwen3' in model_id.lower():
        prompt = prompt + '\n/no_think'

    if getattr(tokenizer, 'chat_template', None):
        text = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = prompt

    inputs = tokenizer(text, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}
    do_sample = temperature is not None and temperature > 0

    generation_kwargs = {
        'max_new_tokens': maxTokens,
        'do_sample': do_sample,
        'pad_token_id': tokenizer.eos_token_id,
        'eos_token_id': tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs['temperature'] = temperature

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **generation_kwargs)

    generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    print('Response:', response[:1000])
    return response


def free_model(model_bundle):
    model_id = model_bundle.get('model_id', 'unknown') if isinstance(model_bundle, dict) else 'unknown'
    if isinstance(model_bundle, dict):
        model_bundle.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Model '{model_id}' released from the notebook process.")
'''

loader_code = (
    loader_template
    .replace('__MODEL_NAME_MAP__', pprint.pformat(model_name_map, sort_dicts=False))
    .replace('__LOAD_IN_4BIT__', repr(LOAD_IN_4BIT))
    .replace('__MAX_TOKENS__', str(MAX_TOKENS_PER_RESPONSE))
).lstrip()
loader_path.write_text(loader_code, encoding='utf-8')

print('Patched config.py MODELS:', models_by_family)
print('Patched HF model map:', model_name_map)
print('RESULTS_DIR:', REPO_RESULTS_DIR)
print('Original Ollama loader backup:', backup_path)

Patched config.py MODELS: {'qwen': ['Qwen/Qwen2.5-7B-Instruct'], 'gemma': ['google/gemma-7b-it']}
Patched HF model map: {'Qwen/Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct', 'google/gemma-7b-it': 'google/gemma-7b-it'}
RESULTS_DIR: /content/drive/MyDrive/COMP6242/Project/llm-failure/results/raw
Original Ollama loader backup: /content/drive/MyDrive/COMP6242/Project/llm-failure/models/model_loader_ollama_backup.py


In [ ]:
# Cheap Python syntax/import sanity checks before touching model inference.

import importlib
import py_compile
import traceback

core_files = [
    CODEBASE / 'config.py',
    CODEBASE / 'run_experiment.py',
    CODEBASE / 'models' / 'model_loader.py',
    CODEBASE / 'data' / 'data_loader.py',
    CODEBASE / 'evaluation' / 'scorer.py',
    CODEBASE / 'evaluation' / 'taxonomy.py',
    CODEBASE / 'prompts' / 'zeroShot.py',
    CODEBASE / 'prompts' / 'randomFewShot.py',
    CODEBASE / 'prompts' / 'targetedFewShots.py',
    CODEBASE / 'prompts' / 'errorTargetedPrompt.py',
]

errors = []
for file_path in core_files:
    try:
        py_compile.compile(str(file_path), doraise=True)
        print(f'PASS syntax: {file_path.relative_to(CODEBASE)}')
    except Exception as exc:
        errors.append((str(file_path), exc))
        print(f'FAIL syntax: {file_path.relative_to(CODEBASE)} -> {exc}')

for module_name in ['config', 'data.data_loader', 'models.model_loader', 'evaluation.scorer', 'evaluation.taxonomy', 'prompts.errorTargetedPrompt']:
    try:
        if module_name in sys.modules:
            del sys.modules[module_name]
        importlib.import_module(module_name)
        print(f'PASS import: {module_name}')
    except Exception as exc:
        errors.append((module_name, exc))
        print(f'FAIL import: {module_name}')
        traceback.print_exc()

if errors:
    raise RuntimeError(f'{len(errors)} syntax/import checks failed. Fix these before running smoke tests.')

print('All syntax/import checks passed.')

PASS syntax: config.py
PASS syntax: run_experiment.py
PASS syntax: models/model_loader.py
PASS syntax: data/data_loader.py
PASS syntax: evaluation/scorer.py
PASS syntax: evaluation/taxonomy.py
PASS syntax: prompts/zeroShot.py
PASS syntax: prompts/randomFewShot.py
PASS syntax: prompts/targetedFewShots.py
PASS syntax: prompts/errorTargetedPrompt.py
PASS import: config
PASS import: data.data_loader
PASS import: models.model_loader
PASS import: evaluation.scorer
PASS import: evaluation.taxonomy
PASS import: prompts.errorTargetedPrompt
All syntax/import checks passed.


In [ ]:
# Hugging Face auth and GPU sanity check.
# Add HF_TOKEN as a Colab secret for gated LLaMA/Gemma models.

import os
import torch
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    secret_token = userdata.get('HF_TOKEN')
except Exception:
    secret_token = None

hf_token = HF_TOKEN or secret_token or os.environ.get('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    try:
        print('Logged in to Hugging Face as:', whoami(token=hf_token).get('name'))
    except Exception as exc:
        print('HF login token was set, but whoami check failed:', exc)
else:
    print('No HF token found. Qwen should work if public; LLaMA/Gemma gated models will fail until HF_TOKEN is provided.')

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Colab, choose Runtime > Change runtime type > GPU.')

print('GPU:', torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f'GPU memory: {props.total_memory / 1024**3:.1f} GB')
print('HF cache:', os.environ.get('HF_HOME'))

In [ ]:
# Hugging Face model access preflight. This does not download weights yet.

from huggingface_hub import model_info

primary_model = next((m for m in MODEL_PLAN if m['family'] == PRIMARY_SMOKE_FAMILY), None)
if primary_model is None:
    raise ValueError(f'PRIMARY_SMOKE_FAMILY not found in MODEL_PLAN: {PRIMARY_SMOKE_FAMILY}')

for model_cfg in [m for m in MODEL_PLAN if m['enabled']]:
    model_id = model_cfg['hf_model_id']
    try:
        info = model_info(model_id, token=os.environ.get('HF_TOKEN'))
        gated = getattr(info, 'gated', None)
        print(f"PASS HF access metadata: {model_cfg['family']} -> {model_id}; gated={gated}")
    except Exception as exc:
        print(f"WARN HF access metadata failed: {model_cfg['family']} -> {model_id}")
        print('     This usually means you need to accept the license and provide HF_TOKEN:', exc)

print('Primary smoke model:', primary_model['hf_model_id'])

In [ ]:
# Dataset loading sanity checks. These are cheap and catch missing HF access / unsupported loader names.
# FOLIO is gated on Hugging Face. If your token/account does not have access yet, it is skipped by default.

from data.data_loader import load_benchMark

GATED_DATASET_HELP = {
    'folio': 'https://huggingface.co/datasets/yale-nlp/FOLIO',
}

required_samples = FEW_SHOT_POOL_SIZE + SMOKE_TEST_SAMPLES
RUN_DATASETS = []
DATASET_FAILURES = {}

for dataset_name in DATASETS:
    try:
        samples = load_benchMark(dataset_name, numSamples=required_samples)
        if not isinstance(samples, list):
            raise TypeError(f'{dataset_name} returned {type(samples)}, expected list')
        if len(samples) < required_samples:
            raise ValueError(f'{dataset_name} returned {len(samples)} samples, expected at least {required_samples}')
        first = samples[0]
        missing = {'question', 'answer', 'category'} - set(first)
        if missing:
            raise KeyError(f'{dataset_name} sample is missing keys: {missing}')
        RUN_DATASETS.append(dataset_name)
        print(f'PASS dataset {dataset_name}: {len(samples)} samples; first category={first.get("category")}')
    except Exception as exc:
        DATASET_FAILURES[dataset_name] = str(exc)
        help_url = GATED_DATASET_HELP.get(dataset_name)
        if help_url:
            print(f'SKIP dataset {dataset_name}: {exc}')
            print(f'     Request/accept access, then rerun this cell: {help_url}')
        else:
            print(f'FAIL dataset {dataset_name}: {exc}')

if DATASET_FAILURES and REQUIRE_ALL_DATASETS:
    raise RuntimeError(f'Dataset checks failed and REQUIRE_ALL_DATASETS=True: {DATASET_FAILURES}')
if not RUN_DATASETS:
    raise RuntimeError('No datasets are available. Fix dataset access before running experiments.')
if SMOKE_BENCHMARK not in RUN_DATASETS:
    raise RuntimeError(f'SMOKE_BENCHMARK={SMOKE_BENCHMARK!r} is not available. Choose one of: {RUN_DATASETS}')

print('Datasets available for this runtime:', RUN_DATASETS)
if DATASET_FAILURES:
    print('Skipped datasets:', DATASET_FAILURES)

PASS dataset gsm_symbolic: 22 samples; first category=arithmetic_symbolic
PASS dataset gsm_plus: 22 samples; first category=arithmetic_perturbed
PASS dataset gsm_ic: 22 samples; first category=arithmetic_word_problem
PASS dataset bigbench_hard: 22 samples; first category=logical_deduction
PASS dataset bigbench_hard_tracking: 22 samples; first category=object_tracking
PASS dataset gsm8k: 22 samples; first category=arithmetic_word_problem
PASS dataset folio: 22 samples; first category=formal_logic
Datasets available for this runtime: ['gsm_symbolic', 'gsm_plus', 'gsm_ic', 'bigbench_hard', 'bigbench_hard_tracking', 'gsm8k', 'folio']


In [ ]:
# Direct Hugging Face model response sanity check.
# This downloads/loads the primary 7B model once and verifies GPU inference before the CLI smoke run.

import gc
import importlib
import torch

for module_name in ['models.model_loader']:
    if module_name in sys.modules:
        del sys.modules[module_name]

from models.model_loader import load_model, generate_response, free_model

model_bundle, tokenizer = load_model(primary_model['display_name'])
text = generate_response(model_bundle, 'Answer with only the number: what is 2 + 2?', maxTokens=32)
print('Direct response:', repr(text))
if not text:
    raise RuntimeError('Direct HF sanity check returned an empty response.')

free_model(model_bundle)
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print('Direct model sanity check passed.')

In [ ]:
# Robust full run loop.
# Runs one model-family / dataset / condition combination at a time.
# Writes logs and manifest rows to OUTPUT_ROOT so disconnect recovery is simple.

import csv
import datetime as dt
import json
import shutil
import subprocess
from itertools import product

MANIFEST_JSON = OUTPUT / 'manifest.json'
MANIFEST_CSV = OUTPUT / 'manifest.csv'


def load_manifest():
    if MANIFEST_JSON.exists():
        return json.loads(MANIFEST_JSON.read_text(encoding='utf-8'))
    return []


def write_manifest(rows):
    MANIFEST_JSON.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding='utf-8')
    fieldnames = [
        'run_id', 'family', 'display_name', 'hf_model_id', 'benchmark', 'condition',
        'status', 'started_at', 'finished_at', 'returncode', 'log_path', 'output_files'
    ]
    with MANIFEST_CSV.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            csv_row = dict(row)
            csv_row['output_files'] = ';'.join(row.get('output_files', []))
            writer.writerow(csv_row)


def successful_existing_row(rows, family, benchmark, condition):
    for row in reversed(rows):
        same_combo = (
            row.get('family') == family
            and row.get('benchmark') == benchmark
            and row.get('condition') == condition
            and row.get('status') == 'success'
        )
        files_exist = all(Path(p).exists() for p in row.get('output_files', []))
        if same_combo and files_exist:
            return row
    return None


def running_existing_row(rows, family, benchmark, condition):
    for row in reversed(rows):
        if (
            row.get('family') == family
            and row.get('benchmark') == benchmark
            and row.get('condition') == condition
            and row.get('status') == 'running'
        ):
            return row
    return None


def model_result_prefix(display_name):
    return display_name.replace('/', '_').replace('.', '-')


def file_matches_condition(path, condition):
    try:
        data = json.loads(Path(path).read_text(encoding='utf-8'))
    except Exception:
        return False
    if not data:
        return False
    for item in data:
        if set(item.get('conditions', {}).keys()) != {condition}:
            return False
    return True


def find_resume_file(model_cfg, benchmark, condition):
    benchmark_dir = REPO_RESULTS_DIR / benchmark
    prefix = model_result_prefix(model_cfg['display_name'])
    candidates = sorted(
        benchmark_dir.glob(f'{prefix}__{benchmark}__*.json'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for candidate in candidates:
        if file_matches_condition(candidate, condition):
            return candidate
    return None


def stream_command_to_log(cmd, log_path):
    with log_path.open('w', encoding='utf-8') as log:
        log.write('COMMAND: ' + ' '.join(cmd) + '\n\n')
        log.flush()
        proc = subprocess.Popen(
            cmd,
            cwd=CODEBASE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        return proc.wait()


if not RUN_FULL_EXPERIMENTS:
    print('RUN_FULL_EXPERIMENTS is False. Smoke tests are safe; full experiments are blocked.')
    print('After smoke tests pass, edit the config cell and set RUN_FULL_EXPERIMENTS = True.')
else:
    manifest = load_manifest()
    enabled_models = [m for m in MODEL_PLAN if m['enabled']]
    datasets_to_run = globals().get('RUN_DATASETS', DATASETS)
    print('Running combinations:', len(enabled_models) * len(datasets_to_run) * len(CONDITIONS))
    print('Datasets in this run:', datasets_to_run)

    for model_cfg in enabled_models:
        print(f"\nMODEL FAMILY: {model_cfg['family']} -> {model_cfg['hf_model_id']}")

        for benchmark, condition in product(datasets_to_run, CONDITIONS):
            existing = successful_existing_row(manifest, model_cfg['family'], benchmark, condition)
            if existing and not FORCE_RERUN:
                print(f"SKIP existing success: {model_cfg['family']} / {benchmark} / {condition}")
                continue

            resume_file = None
            row = None
            running = running_existing_row(manifest, model_cfg['family'], benchmark, condition)
            if running and not FORCE_RERUN:
                resume_file = find_resume_file(model_cfg, benchmark, condition)
                if resume_file:
                    row = running
                    run_id = row['run_id']
                    log_path = Path(row['log_path'])
                    status_path = OUTPUT_STATUS_DIR / f'{run_id}.json'
                    print(f"RESUME existing running combo: {model_cfg['family']} / {benchmark} / {condition}")
                    print(f"Resume file: {resume_file}")
                else:
                    print(f"Found running manifest row for {model_cfg['family']} / {benchmark} / {condition}, but no partial JSON was found. Starting a new file.")

            if row is None:
                timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
                run_id = f"{model_cfg['family']}__{benchmark}__{condition}__{timestamp}"
                log_path = OUTPUT_LOG_DIR / f'{run_id}.log'
                status_path = OUTPUT_STATUS_DIR / f'{run_id}.json'
                row = {
                    'run_id': run_id,
                    'family': model_cfg['family'],
                    'display_name': model_cfg['display_name'],
                    'hf_model_id': model_cfg['hf_model_id'],
                    'benchmark': benchmark,
                    'condition': condition,
                    'status': 'running',
                    'started_at': dt.datetime.now().isoformat(timespec='seconds'),
                    'finished_at': '',
                    'returncode': None,
                    'log_path': str(log_path),
                    'output_files': [],
                }
                manifest.append(row)

            cmd = [
                sys.executable,
                str(CODEBASE / 'run_experiment.py'),
                '--model', model_cfg['family'],
                '--benchmark', benchmark,
                '--condition', condition,
            ]
            if resume_file:
                cmd.extend(['--resume-file', str(resume_file)])

            row['status'] = 'running'
            row['finished_at'] = ''
            row['returncode'] = None
            row['log_path'] = str(log_path)
            write_manifest(manifest)
            status_path.write_text(json.dumps(row, indent=2), encoding='utf-8')

            before = set(REPO_RESULTS_DIR.rglob('*.json'))
            print(f"\nSTART {row['run_id']}")
            returncode = stream_command_to_log(cmd, log_path)
            after = set(REPO_RESULTS_DIR.rglob('*.json'))
            new_files = sorted(after - before, key=lambda p: p.stat().st_mtime)
            if resume_file and Path(resume_file).exists() and Path(resume_file) not in new_files:
                new_files.append(Path(resume_file))

            copied = []
            for src in new_files:
                rel_name = src.relative_to(REPO_RESULTS_DIR).as_posix().replace('/', '__')
                dest = OUTPUT_RAW_DIR / f"{row['run_id']}__{rel_name}"
                shutil.copy2(src, dest)
                copied.append(str(dest))
                print(f'Copied result: {dest}')

            row['finished_at'] = dt.datetime.now().isoformat(timespec='seconds')
            row['returncode'] = returncode
            row['output_files'] = copied
            row['status'] = 'success' if returncode == 0 and copied else 'failed'
            write_manifest(manifest)
            status_path.write_text(json.dumps(row, indent=2), encoding='utf-8')

            if row['status'] != 'success':
                print(f"FAILED {row['run_id']}; see {log_path}")
                raise RuntimeError(f"Stopping after failed run: {row['run_id']}")

            print(f"DONE {row['run_id']}")

    print('All requested full-run combinations completed or skipped.')

Streaming output truncated to the last 5000 lines.
* The fifth statement is also false. It is not necessarily true that Vladimir is a manager at Gazprom.

**Step 2: Conclusion:**

**Answer:** Uncertain

The statement "Vladimir is a Russian federation official" is uncertain because it is not clear whether he holds Taiwanese citizenship or not.
  Saved: /content/drive/MyDrive/COMP6242/Project/llm-failure/results/raw/folio/google_gemma-7b-it__folio__20260527_2233.json
[126 / 183] Evaluting Question
Response: **Step 1:** Identify the relevant information.

- The statement "Everyone who can register to vote in the United States can participate in the 2024 United States presidential election" is true.
- The statement "If someone has United States citizenship, then they can register to vote in the United States" is true.
- The statement "A person either has United States citizenship or Taiwanese citizenship" is true.
- The statement "No Russian Federation officials hold Taiwanese citizenship"

In [ ]:
# Compact result summary from persistent OUTPUT_ROOT/raw JSON files.

import json
from collections import defaultdict
from pathlib import Path
import pandas as pd

manifest_rows = load_manifest() if MANIFEST_JSON.exists() else []
summary_rows = []

for row in manifest_rows:
    if row.get('status') != 'success':
        continue
    for output_file in row.get('output_files', []):
        path = Path(output_file)
        if not path.exists():
            continue
        data = json.loads(path.read_text(encoding='utf-8'))
        total = len(data)
        correct = 0
        for item in data:
            condition_block = item.get('conditions', {}).get(row.get('condition'), {})
            if condition_block.get('majority_correct'):
                correct += 1
        summary_rows.append({
            'family': row.get('family'),
            'model': row.get('display_name'),
            'hf_model_id': row.get('hf_model_id'),
            'benchmark': row.get('benchmark'),
            'condition': row.get('condition'),
            'n': total,
            'correct': correct,
            'accuracy': (correct / total) if total else None,
            'result_file': str(path),
        })

# Include smoke outputs if no manifest success exists yet.
if not summary_rows:
    for path in sorted(OUTPUT_RAW_DIR.glob('*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        by_condition = defaultdict(lambda: {'n': 0, 'correct': 0})
        for item in data:
            for condition, block in item.get('conditions', {}).items():
                by_condition[condition]['n'] += 1
                by_condition[condition]['correct'] += int(bool(block.get('majority_correct')))
        for condition, stats in by_condition.items():
            summary_rows.append({
                'family': 'unknown_or_smoke',
                'model': 'unknown_or_smoke',
                'hf_model_id': 'unknown_or_smoke',
                'benchmark': 'unknown_or_smoke',
                'condition': condition,
                'n': stats['n'],
                'correct': stats['correct'],
                'accuracy': (stats['correct'] / stats['n']) if stats['n'] else None,
                'result_file': str(path),
            })

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT / 'accuracy_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'Saved summary CSV: {summary_csv}')
summary_df

Saved summary CSV: /content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results/accuracy_summary.csv


,family,model,hf_model_id,benchmark,condition,n,correct,accuracy,result_file
0,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,zero_shot_baseline,200,166,0.830000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
1,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,zero_shot,200,107,0.535000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
2,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,targeted_few_shot_answer_only,200,125,0.625000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
3,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,random_few_shot,200,143,0.715000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
4,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,targeted_few_shot,200,148,0.740000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
...,...,...,...,...,...,...,...,...,...
135,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,targeted_few_shot_k5,183,76,0.415301,/content/drive/MyDrive/COMP6242/Project/llm-fa...
136,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,error_targeted_icl,183,85,0.464481,/content/drive/MyDrive/COMP6242/Project/llm-fa...
137,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,error_targeted_icl_random,183,86,0.469945,/content/drive/MyDrive/COMP6242/Project/llm-fa...
138,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,error_targeted_icl_correct_only,183,91,0.497268,/content/drive/MyDrive/COMP6242/Project/llm-fa...


In [ ]:
# Report-oriented metrics from saved raw JSON files.
# Post-processing only: rerun this later without rerunning inference.

import json
import math
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

BASELINE_CONDITION = 'zero_shot_baseline'
FALLBACK_BASELINE_CONDITION = 'zero_shot'

manifest_rows = load_manifest() if MANIFEST_JSON.exists() else []
result_records = []

for row in manifest_rows:
    if row.get('status') != 'success':
        continue
    for output_file in row.get('output_files', []):
        path = Path(output_file)
        if path.exists():
            result_records.append({**row, 'path': path})

if not result_records:
    for path in sorted(OUTPUT_RAW_DIR.glob('*.json')):
        result_records.append({
            'family': 'unknown_or_smoke',
            'display_name': 'unknown_or_smoke',
            'hf_model_id': 'unknown_or_smoke',
            'benchmark': 'unknown_or_smoke',
            'condition': None,
            'path': path,
        })

items_by_combo = {}
summary_rows = []
for record in result_records:
    data = json.loads(record['path'].read_text(encoding='utf-8'))
    file_conditions = sorted({cond for item in data for cond in item.get('conditions', {})})
    conditions_to_score = [record.get('condition')] if record.get('condition') else file_conditions

    for condition in conditions_to_score:
        if not condition:
            continue
        keyed_items = {}
        correct = 0
        for idx, item in enumerate(data):
            block = item.get('conditions', {}).get(condition, {})
            if not block:
                continue
            is_correct = bool(block.get('majority_correct'))
            correct += int(is_correct)
            runs = block.get('runs', [])
            first_run = runs[0] if runs else {}
            key = item.get('sample_id', item.get('question', idx))
            keyed_items[key] = {
                'question': item.get('question'),
                'category': item.get('benchmark_category'),
                'correct': is_correct,
                'error_type': first_run.get('error_type'),
                'predicted': first_run.get('predicted'),
                'ground_truth': item.get('ground_truth'),
            }

        total = len(keyed_items)
        combo = (record.get('family'), record.get('benchmark'), condition)
        items_by_combo[combo] = keyed_items
        summary_rows.append({
            'family': record.get('family'),
            'model': record.get('display_name'),
            'hf_model_id': record.get('hf_model_id'),
            'benchmark': record.get('benchmark'),
            'condition': condition,
            'n': total,
            'correct': correct,
            'accuracy': (correct / total) if total else None,
            'result_file': str(record['path']),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT / 'accuracy_summary.csv', index=False)

error_rows = []
for (family, benchmark, condition), items in items_by_combo.items():
    counts = Counter(v.get('error_type') or ('correct' if v.get('correct') else 'unknown') for v in items.values())
    total = sum(counts.values())
    for error_type, count in counts.items():
        error_rows.append({
            'family': family,
            'benchmark': benchmark,
            'condition': condition,
            'error_type': error_type,
            'count': count,
            'pct': count / total if total else None,
        })
error_df = pd.DataFrame(error_rows)
error_df.to_csv(OUTPUT / 'error_distribution.csv', index=False)

recovery_rows = []
for family in sorted({k[0] for k in items_by_combo}):
    family_benchmarks = sorted({k[1] for k in items_by_combo if k[0] == family})
    for benchmark in family_benchmarks:
        baseline_key = (family, benchmark, BASELINE_CONDITION)
        if baseline_key not in items_by_combo:
            baseline_key = (family, benchmark, FALLBACK_BASELINE_CONDITION)
        baseline_items = items_by_combo.get(baseline_key, {})
        baseline_failures = {k: v for k, v in baseline_items.items() if not v.get('correct')}
        conditions = sorted({k[2] for k in items_by_combo if k[0] == family and k[1] == benchmark})
        for condition in conditions:
            if condition == baseline_key[2]:
                continue
            target_items = items_by_combo.get((family, benchmark, condition), {})
            by_error = defaultdict(lambda: {'n': 0, 'recovered': 0})
            for key, base in baseline_failures.items():
                if key not in target_items:
                    continue
                error_type = base.get('error_type') or 'unknown'
                by_error[error_type]['n'] += 1
                by_error[error_type]['recovered'] += int(bool(target_items[key].get('correct')))
            for error_type, stats in by_error.items():
                recovery_rows.append({
                    'family': family,
                    'benchmark': benchmark,
                    'baseline_condition': baseline_key[2],
                    'condition': condition,
                    'error_type': error_type,
                    'n_baseline_failures': stats['n'],
                    'recovered': stats['recovered'],
                    'recovery_rate': stats['recovered'] / stats['n'] if stats['n'] else None,
                })
recovery_df = pd.DataFrame(recovery_rows)
recovery_df.to_csv(OUTPUT / 'per_error_recovery.csv', index=False)

def js_divergence(counts_a, counts_b):
    labels = sorted(set(counts_a) | set(counts_b))
    total_a = sum(counts_a.values())
    total_b = sum(counts_b.values())
    if not total_a or not total_b:
        return None
    pa = [counts_a.get(label, 0) / total_a for label in labels]
    pb = [counts_b.get(label, 0) / total_b for label in labels]
    pm = [(a + b) / 2 for a, b in zip(pa, pb)]
    def kl(p, q):
        return sum(x * math.log2(x / y) for x, y in zip(p, q) if x > 0 and y > 0)
    return 0.5 * kl(pa, pm) + 0.5 * kl(pb, pm)

js_rows = []
mcnemar_rows = []
for family in sorted({k[0] for k in items_by_combo}):
    for benchmark in sorted({k[1] for k in items_by_combo if k[0] == family}):
        baseline_key = (family, benchmark, BASELINE_CONDITION)
        if baseline_key not in items_by_combo:
            baseline_key = (family, benchmark, FALLBACK_BASELINE_CONDITION)
        baseline_items = items_by_combo.get(baseline_key, {})
        baseline_counts = Counter(v.get('error_type') or ('correct' if v.get('correct') else 'unknown') for v in baseline_items.values())
        for condition in sorted({k[2] for k in items_by_combo if k[0] == family and k[1] == benchmark}):
            if condition == baseline_key[2]:
                continue
            target_items = items_by_combo.get((family, benchmark, condition), {})
            target_counts = Counter(v.get('error_type') or ('correct' if v.get('correct') else 'unknown') for v in target_items.values())
            js_rows.append({
                'family': family,
                'benchmark': benchmark,
                'baseline_condition': baseline_key[2],
                'condition': condition,
                'js_divergence': js_divergence(baseline_counts, target_counts),
            })

            b = c = paired = 0
            for key, base in baseline_items.items():
                if key not in target_items:
                    continue
                paired += 1
                base_correct = bool(base.get('correct'))
                target_correct = bool(target_items[key].get('correct'))
                b += int((not base_correct) and target_correct)
                c += int(base_correct and (not target_correct))
            chi2 = ((abs(b - c) - 1) ** 2 / (b + c)) if (b + c) else None
            p_value = math.erfc(math.sqrt(chi2 / 2)) if chi2 is not None else None
            mcnemar_rows.append({
                'family': family,
                'benchmark': benchmark,
                'baseline_condition': baseline_key[2],
                'condition': condition,
                'paired_n': paired,
                'baseline_wrong_target_right': b,
                'baseline_right_target_wrong': c,
                'mcnemar_chi2_continuity': chi2,
                'p_value_approx': p_value,
            })

js_df = pd.DataFrame(js_rows)
js_df.to_csv(OUTPUT / 'js_divergence.csv', index=False)
mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(OUTPUT / 'mcnemar_tests.csv', index=False)

robustness_rows = []
if not summary_df.empty:
    for (family, condition), group in summary_df.groupby(['family', 'condition']):
        clean = group[group['benchmark'] == 'gsm8k']
        if clean.empty or not clean.iloc[0]['accuracy']:
            continue
        clean_acc = clean.iloc[0]['accuracy']
        for _, row in group.iterrows():
            if row['benchmark'] == 'gsm8k':
                continue
            robustness_rows.append({
                'family': family,
                'condition': condition,
                'clean_benchmark': 'gsm8k',
                'benchmark': row['benchmark'],
                'clean_accuracy': clean_acc,
                'benchmark_accuracy': row['accuracy'],
                'robustness_ratio': row['accuracy'] / clean_acc if clean_acc else None,
            })
robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(OUTPUT / 'robustness_ratio.csv', index=False)

for filename in [
    'accuracy_summary.csv',
    'error_distribution.csv',
    'per_error_recovery.csv',
    'js_divergence.csv',
    'mcnemar_tests.csv',
    'robustness_ratio.csv',
]:
    print('Saved:', OUTPUT / filename)

print('These metrics are generated from saved raw JSON only; no model inference is run in this cell.')
summary_df

Saved: /content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results/accuracy_summary.csv
Saved: /content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results/error_distribution.csv
Saved: /content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results/per_error_recovery.csv
Saved: /content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results/js_divergence.csv
Saved: /content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results/mcnemar_tests.csv
Saved: /content/drive/MyDrive/COMP6242/Project/llm-failure/llm-failure-7b-results/robustness_ratio.csv
These metrics are generated from saved raw JSON only; no model inference is run in this cell.


,family,model,hf_model_id,benchmark,condition,n,correct,accuracy,result_file
0,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,zero_shot_baseline,200,166,0.830000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
1,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,zero_shot,200,107,0.535000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
2,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,targeted_few_shot_answer_only,200,125,0.625000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
3,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,random_few_shot,200,143,0.715000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
4,qwen,Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-7B-Instruct,gsm_symbolic,targeted_few_shot,200,148,0.740000,/content/drive/MyDrive/COMP6242/Project/llm-fa...
...,...,...,...,...,...,...,...,...,...
135,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,targeted_few_shot_k5,183,76,0.415301,/content/drive/MyDrive/COMP6242/Project/llm-fa...
136,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,error_targeted_icl,183,85,0.464481,/content/drive/MyDrive/COMP6242/Project/llm-fa...
137,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,error_targeted_icl_random,183,86,0.469945,/content/drive/MyDrive/COMP6242/Project/llm-fa...
138,gemma,google/gemma-7b-it,google/gemma-7b-it,folio,error_targeted_icl_correct_only,183,91,0.497268,/content/drive/MyDrive/COMP6242/Project/llm-fa...


## Recovery / Resume Notes

If Colab disconnects:

1. Reconnect to a GPU runtime.
2. Run the Drive mount, editable config, path setup, dependency install, runtime patch, HF auth/GPU check, and access preflight cells again.
3. Run the dataset and direct model sanity checks if the runtime is fresh.
4. Run the full-run cell again. It reads `manifest.json` in `OUTPUT_ROOT` and skips completed combinations when `FORCE_RERUN = False`.

Where outputs go:

- Repo-side incremental raw JSONs: `CODEBASE_DIR/results/raw/<benchmark>/`
- Persistent copied raw JSONs: `OUTPUT_ROOT/raw`
- Per-run logs: `OUTPUT_ROOT/logs`
- Manifest: `OUTPUT_ROOT/manifest.json` and `OUTPUT_ROOT/manifest.csv`
- Per-run status snapshots: `OUTPUT_ROOT/status`
- Accuracy summary: `OUTPUT_ROOT/accuracy_summary.csv`
- HF cache: `MODEL_CACHE_DIR`

Before spending credits, confirm that the smoke JSON appears in `OUTPUT_ROOT/raw` and that the summary cell displays at least one row. If LLaMA or Gemma fail during access/model loading, accept their licenses on Hugging Face and set `HF_TOKEN` as a Colab secret.